In [0]:
# ============================================================
# 03_silver_dallas  (rewritten)
# ============================================================

# Source  : bronze.dallas_raw (latest _batch_id)
# Targets : silver.dallas_clean
#           silver.dallas_violations
#           silver.dallas_quarantine

# Write mode: OVERWRITE (full refresh)
# ============================================================


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType
from functools import reduce

CATALOG = "final_project_databi_group8"

BRONZE_TABLE = f"{CATALOG}.bronze.dallas_raw"
SILVER_TABLE = f"{CATALOG}.silver.dallas_clean"
SILVER_VIOLS = f"{CATALOG}.silver.dallas_violations"
QUARANTINE   = f"{CATALOG}.silver.dallas_quarantine"

DAL_ZIP_MIN   = 75001
DAL_ZIP_MAX   = 79999
DAL_SCORE_MIN = 0
DAL_SCORE_MAX = 100

# Defining violation wide columns here so all cells can access it
VIOL_WIDE_COLS = (
    [f"viol_desc_{i}"   for i in range(1, 26)] +
    [f"viol_points_{i}" for i in range(1, 26)] +
    [f"viol_detail_{i}" for i in range(1, 26)] +
    [f"viol_memo_{i}"   for i in range(1, 26)]
)

print(f"Catalog       : {CATALOG}")
print(f"Bronze source : {BRONZE_TABLE}")
print(f"Silver target : {SILVER_TABLE}")

In [0]:
# Read latest batch
try:
    spark.table(BRONZE_TABLE).limit(1).collect()
    print(f"Bronze table found: {BRONZE_TABLE}")
except Exception as e:
    raise Exception(f"Bronze table not found: {BRONZE_TABLE}\nError: {e}")

latest_batch = (
    spark.table(BRONZE_TABLE)
    .groupBy("_batch_id")
    .agg(F.max("_extract_ts").alias("ts"))
    .orderBy(F.col("ts").desc())
    .select("_batch_id")
    .first()[0]
)
print(f"Latest batch : {latest_batch}")

DAL_CORE = [
    "inspection_id", "restaurant_name", "inspection_type",
    "inspection_date", "inspection_score",
    "street_address", "zip_code", "location_raw",
    "city_source", "_batch_id",
    F.col("_extract_ts").alias("_bronze_extract_ts")
]

VIOL_COLS = (
    [f"viol_desc_{i}"   for i in range(1, 26)] +
    [f"viol_points_{i}" for i in range(1, 26)] +
    [f"viol_detail_{i}" for i in range(1, 26)] +
    [f"viol_memo_{i}"   for i in range(1, 26)]
)

df_dal = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("_batch_id") == latest_batch)
    .select(DAL_CORE + VIOL_COLS)
    .dropDuplicates(["inspection_id"])
)

print(f"Rows after dedup : {df_dal.count():,}")
print(f"Columns          : {len(df_dal.columns)}")


In [0]:
df_dal_typed = (
    df_dal
    .withColumn("inspection_score",
        F.col("inspection_score").cast(DoubleType()))
    .withColumn("inspection_date",
        F.coalesce(
            F.try_to_date(F.col("inspection_date"), F.lit("MM/dd/yyyy")),
            F.try_to_date(F.col("inspection_date"), F.lit("MM-dd-yyyy")),
            F.try_to_date(F.col("inspection_date"), F.lit("dd-MM-yyyy")),
            F.try_to_date(F.col("inspection_date"), F.lit("yyyy-MM-dd"))
        ))
)

# Zip cleaning
df_dal_typed = df_dal_typed.withColumn(
    "zip_code",
    F.regexp_extract(F.col("zip_code"), r"^(\d{5})", 1)
)
df_dal_typed = df_dal_typed.withColumn(
    "zip_code",
    F.when(F.col("zip_code") == "", F.lit(None))
     .otherwise(F.col("zip_code"))
)

# Lat/Lon split from location_raw
df_dal_typed = (
    df_dal_typed
    .withColumn("latitude",
        F.regexp_extract(F.col("location_raw"),
            r"\(([0-9.\-]+),", 1).cast(DoubleType()))
    .withColumn("longitude",
        F.regexp_extract(F.col("location_raw"),
            r",\s*([0-9.\-]+)\)", 1).cast(DoubleType()))
)

# Cast per-violation points to DOUBLE
for i in range(1, 26):
    df_dal_typed = df_dal_typed.withColumn(
        f"viol_points_{i}",
        F.col(f"viol_points_{i}").cast(DoubleType())
    )

null_dates = df_dal_typed.filter(F.col("inspection_date").isNull()).count()
print(f"Null dates after casting : {null_dates}")
print("Type casting complete")

display(df_dal_typed.select(
    "inspection_id", "inspection_date", "inspection_score",
    "zip_code", "latitude", "longitude"
).limit(3))

In [0]:
# Field standardisation
# Add columns required for Silver Union with Chicago.

df_dal_typed = (
    df_dal_typed
    # Dallas has no city/state columns — add literals
    .withColumn("city",              F.lit("Dallas"))
    .withColumn("state",             F.lit("TX"))
    # NULL schema columns for Chicago-only fields
    # Typed correctly so UNION ALL with Chicago works
    .withColumn("aka_name",          F.lit(None).cast("string"))
    .withColumn("license_number",    F.lit(None).cast("string"))
    .withColumn("facility_type",     F.lit(None).cast("string"))
    .withColumn("risk_category",     F.lit(None).cast("string"))
    .withColumn("inspection_result", F.lit(None).cast("string"))
    # Count non-null violation slots for DQX check below
    .withColumn("raw_viol_count",
        sum(
            F.when(
                F.col(f"viol_desc_{i}").isNotNull() &
                (F.col(f"viol_desc_{i}") != ""), 1
            ).otherwise(0)
            for i in range(1, 26)
        ))
)

# Drop redundant derived columns from source inspection_month ('Oct 2016') and inspection_year ('FY2017') duplicate what inspection_date already encodes location_raw has been split into latitude/longitude

for col in ["inspection_month", "inspection_year", "location_raw"]:
    if col in df_dal_typed.columns:
        df_dal_typed = df_dal_typed.drop(col)

print("Standardisation complete")
display(df_dal_typed.select(
    "inspection_id", "city", "state", "risk_category",
    "inspection_result", "raw_viol_count"
).limit(3))


In [0]:
# Step 4: DQX Validation
df_dal_checked = (
    df_dal_typed

    .withColumn("valid_business_name",
        F.col("restaurant_name").isNotNull() &
        (F.col("restaurant_name") != ""))

    .withColumn("valid_inspection_date",
        F.col("inspection_date").isNotNull())

    .withColumn("valid_inspection_type",
        F.col("inspection_type").isNotNull() &
        (F.col("inspection_type") != ""))

    .withColumn("valid_zip_code",
        F.col("zip_code").isNotNull() &
        (F.col("zip_code") != "") &
        (F.col("zip_code") != "null"))

    .withColumn("valid_zip_format",
        F.when(F.col("zip_code").isNull(), F.lit(False))
         .otherwise(F.col("zip_code").rlike(r"^[0-9]+$")))

    # Dallas TX zip range 75001-79999
    .withColumn("valid_zip_range",
        F.when(F.col("zip_code").isNull(), F.lit(False))
         .otherwise(F.col("zip_code").cast("int").between(
             DAL_ZIP_MIN, DAL_ZIP_MAX)))

    # Score must be 0-100 (6 records have negative scores)
    .withColumn("valid_score_range",
        F.col("inspection_score").isNotNull() &
        F.col("inspection_score").between(DAL_SCORE_MIN, DAL_SCORE_MAX))

    # At least 1 violation slot filled
    .withColumn("valid_has_violations",
        F.col("raw_viol_count") >= 1)

    # Score >= 90 must have at most 3 violations
    .withColumn("valid_score_viol_rule",
        ~((F.col("inspection_score") >= 90) &
          (F.col("raw_viol_count") > 3)))

    .withColumn("_all_checks_pass",
        F.col("valid_business_name")    &
        F.col("valid_inspection_date")  &
        F.col("valid_inspection_type")  &
        F.col("valid_zip_code")         &
        F.col("valid_zip_format")       &
        F.col("valid_zip_range")        &
        F.col("valid_score_range")      &
        F.col("valid_has_violations")   &
        F.col("valid_score_viol_rule"))
)

CHECK_COLS = [
    "valid_business_name", "valid_inspection_date",
    "valid_inspection_type", "valid_zip_code", "valid_zip_format",
    "valid_zip_range", "valid_score_range",
    "valid_has_violations", "valid_score_viol_rule", "_all_checks_pass"
]

failed_checks_expr = F.concat_ws(",", *[
    F.when(F.col(c) == False, F.lit(c)).otherwise(F.lit(None))
    for c in CHECK_COLS if c != "_all_checks_pass"
])
df_dal_checked = df_dal_checked.withColumn(
    "_dqx_failed_checks", failed_checks_expr
)

df_dal_good = (
    df_dal_checked
    .filter(F.col("_all_checks_pass") == True)
    .drop(*CHECK_COLS, "_dqx_failed_checks", "raw_viol_count")
    .withColumn("_silver_ts",      F.current_timestamp())
    .withColumn("_silver_date",    F.current_date().cast("string"))
    .withColumn("_silver_version", F.lit("v1"))
    .withColumn("_dqx_status",     F.lit("PASS"))
)

df_dal_bad = (
    df_dal_checked
    .filter(
        F.col("_all_checks_pass").isNull() |
        (F.col("_all_checks_pass") == False))
    .withColumn("_silver_ts",      F.current_timestamp())
    .withColumn("_silver_date",    F.current_date().cast("string"))
    .withColumn("_silver_version", F.lit("v1"))
    .withColumn("_dqx_status",     F.lit("FAIL"))
)

original   = df_dal_typed.count()
good_count = df_dal_good.count()
bad_count  = df_dal_bad.count()
lost       = original - (good_count + bad_count)

print(f"Original rows : {original:,}")
print(f"GOOD rows     : {good_count:,}")
print(f"BAD rows      : {bad_count:,}")
print(f"Rows lost     : {lost}")
assert lost == 0, f"Lost {lost} rows - logic error in DQX split"

print("\nFailed check breakdown:")
display(
    df_dal_bad
    .groupBy("_dqx_failed_checks")
    .count()
    .orderBy(F.desc("count"))
    .limit(15)
)


In [0]:
# Violation unpivot 
# Dallas stores violations in 25 slots of 4 sub-fields each.

dfs = []
for i in range(1, 26):
    dfs.append(
        df_dal_good.select(
            "inspection_id", "restaurant_name",
            "inspection_date", "city_source", "_batch_id",
            F.col(f"viol_desc_{i}").alias("violation_description"),
            F.col(f"viol_points_{i}").alias("violation_points"),
            F.col(f"viol_detail_{i}").alias("violation_detail"),
            F.col(f"viol_memo_{i}").alias("inspector_comment"),
            F.lit(i).cast(IntegerType()).alias("violation_slot")
        )
        .filter(
            F.col("violation_description").isNotNull() &
            (F.col("violation_description") != ""))
    )

df_dal_long = reduce(lambda a, b: a.union(b), dfs)

# Extract violation code
df_dal_long = df_dal_long.withColumn(
    "violation_code",
    F.regexp_extract(F.col("violation_description"), r"^\*?(\d+)", 1)
)

# Dedup: one row per inspection + violation code
df_dal_long = df_dal_long.dropDuplicates(
    ["inspection_id", "violation_code"]
)

# Count and join back
viol_counts = (
    df_dal_long
    .groupBy("inspection_id")
    .agg(F.countDistinct("violation_code").alias("violation_count"))
)

df_dal_good = (
    df_dal_good.alias("g")
    .join(viol_counts.alias("v"), on="inspection_id", how="left")
    .withColumn("violation_count",
        F.coalesce(F.col("v.violation_count"), F.lit(0)).cast(IntegerType()))
    .select(
        *[F.col(f"g.{c}") for c in df_dal_good.columns],
        F.col("violation_count")
    )
)

# Drop the 100 wide violation columns
df_dal_final = df_dal_good.drop(
    *[c for c in VIOL_WIDE_COLS if c in df_dal_good.columns]
)

print(f"Dallas clean inspections (wide cols dropped) : {df_dal_final.count():,}")
print(f"Dallas violation rows (long format)          : {df_dal_long.count():,}")
display(df_dal_long.select(
    "inspection_id", "violation_code", "violation_description",
    "violation_points", "inspector_comment"
).limit(5))


In [0]:
# Write all three outputs
(
    df_dal_final
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)
print(f"{SILVER_TABLE}  ({df_dal_final.count():,} rows, {len(df_dal_final.columns)} cols)")

(
    df_dal_long
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_VIOLS)
)
print(f"{SILVER_VIOLS}  ({df_dal_long.count():,} rows)")

# Drop wide cols from bad rows too before writing quarantine
cols_to_drop_bad = [c for c in VIOL_WIDE_COLS if c in df_dal_bad.columns]
df_dal_bad_final = df_dal_bad.drop(*cols_to_drop_bad)

(
    df_dal_bad_final
    .write.format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(QUARANTINE)
)
print(f"{QUARANTINE}  ({df_dal_bad_final.count():,} rows)")


In [0]:
# Verification
display(spark.sql(f"""
    SELECT
        city_source,
        COUNT(*)                              AS total_inspections,
        ROUND(AVG(inspection_score), 2)       AS avg_score,
        ROUND(AVG(violation_count), 2)        AS avg_violations,
        MIN(inspection_score)                 AS min_score,
        MAX(inspection_score)                 AS max_score,
        SUM(CASE WHEN inspection_type = 'Routine'   THEN 1 ELSE 0 END) AS routine,
        SUM(CASE WHEN inspection_type = 'Follow-up' THEN 1 ELSE 0 END) AS followup,
        SUM(CASE WHEN inspection_type = 'Complaint' THEN 1 ELSE 0 END) AS complaint
    FROM {SILVER_TABLE}
    GROUP BY city_source
"""))

display(spark.sql(f"""
    SELECT
        SUM(CASE WHEN valid_business_name    = false THEN 1 ELSE 0 END) AS failed_name,
        SUM(CASE WHEN valid_inspection_date  = false THEN 1 ELSE 0 END) AS failed_date,
        SUM(CASE WHEN valid_inspection_type  = false THEN 1 ELSE 0 END) AS failed_type,
        SUM(CASE WHEN valid_zip_code         = false THEN 1 ELSE 0 END) AS failed_zip_null,
        SUM(CASE WHEN valid_zip_format       = false THEN 1 ELSE 0 END) AS failed_zip_fmt,
        SUM(CASE WHEN valid_zip_range        = false THEN 1 ELSE 0 END) AS failed_zip_range,
        SUM(CASE WHEN valid_score_range      = false THEN 1 ELSE 0 END) AS failed_score,
        SUM(CASE WHEN valid_has_violations   = false THEN 1 ELSE 0 END) AS failed_no_viols,
        SUM(CASE WHEN valid_score_viol_rule  = false THEN 1 ELSE 0 END) AS failed_score_viol,
        COUNT(*)                                                          AS total_quarantined
    FROM {QUARANTINE}
"""))

display(spark.sql(f"DESCRIBE HISTORY {SILVER_TABLE}"))
